# Исследование территории: Фрунзенский район Санкт-Петербурга

Этот ноутбук считает городские показатели Фрунзенского района с помощью библиотеки
**BlocksNet 1.0.0a9**. Порядок работы:

1. скачивает готовые городские данные и готовый граф дорог;
2. выделяет кварталы Фрунзенского района (и буферную зону вокруг);
3. считает **время в пути на автомобиле между кварталами** — в минутах;
4. считает **плотность застройки** и **морфотипы** (типы застройки);
5. считает **обеспеченность** базовым набором услуг;
6. собирает всё в одну таблицу и сохраняет результаты в папку `outputs/`.

### Источники данных

- **Кварталы, здания, услуги** — открытый набор города: https://github.com/IDUclub/blocksnet-data,
  релиз `v0.0.1`, файл `saint_petersburg.zip` (около 400 МБ). При первом запуске он скачается сам.
- **Граф дорог** — `graph_drive.pickle` из того же архива. Это готовый граф **для личного
  автотранспорта**: вес ребра — **время в пути в минутах** (атрибут `time_min`).
  Интермодальный (транспорт + пешком) граф мы **не строим**.
- **Граница района** — OpenStreetMap, административная граница Фрунзенского района (relation 369514).

> ⚠️ Ноутбук сам ничего не считает при открытии: вы запускаете ячейки по порядку сверху вниз.

## Как запустить (Windows)

1. Откройте терминал в корне проекта (там, где лежит `pyproject.toml`).
2. Убедитесь, что зависимости установлены:
   ```
   .venv\Scripts\python.exe -m pip install -e ".[dev]"
   ```
   Если BlocksNet ещё не установлен:
   ```
   .venv\Scripts\python.exe -m pip install blocksnet==1.0.0a9
   ```
3. Запустите Jupyter:
   ```
   .venv\Scripts\jupyter.exe notebook
   ```
4. Откройте файл `examples/frunzensky_blocksnet.ipynb` и выполняйте ячейки сверху вниз.

In [ ]:
# === 0. Настройка окружения и папок ===
# Все пути считаются от корня проекта (папки, где лежит pyproject.toml).
import json
import pickle
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt

import blocksnet

print("BlocksNet:", blocksnet.__version__)
print("GeoPandas:", gpd.__version__)

# Корень проекта: ноутбук лежит в examples/, поэтому при запуске из examples поднимаемся на уровень выше
if Path.cwd().name == "examples":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
CITY_DIR = DATA_DIR / "saint_petersburg"
OUT_DIR = PROJECT_DIR / "outputs"
for folder in (DATA_DIR, CITY_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Папка данных:     ", CITY_DIR)
print("Папка результатов:", OUT_DIR)

## 1. Данные города

Скачиваем и распаковываем архив с кварталами, зданиями, услугами и графами.

**Важно:** эта ячейка качает около 400 МБ только при первом запуске. Если архив уже есть,
она просто его пропустит (повторный запуск не качает данные заново).

In [ ]:
# === 1.1. Скачивание архива с данными (один раз) ===
import requests

# Готовые данные города (подготовлены скриптом cities/saint_petersburg/main.py)
ZIP_URL = "https://github.com/IDUclub/blocksnet-data/releases/download/v0.0.1/saint_petersburg.zip"
ZIP_PATH = DATA_DIR / "saint_petersburg.zip"

if ZIP_PATH.exists():
    print(f"Архив уже скачан: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size / 1e6:.0f} МБ)")
else:
    print("Скачиваем архив (около 400 МБ). Это может занять несколько минут...")
    with requests.get(ZIP_URL, stream=True, timeout=180) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))
        done = 0
        with open(ZIP_PATH, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r{done / 1e6:7.1f} из {total / 1e6:.0f} МБ", end="")
    print(f"\nГотово: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.0f} МБ)")

In [ ]:
# === 1.2. Распаковка архива (один раз) ===
BLOCKS_FILE = CITY_DIR / "blocks.geojson"
GRAPH_FILE = CITY_DIR / "graph_drive.pickle"

if BLOCKS_FILE.exists():
    print("Данные уже распакованы.")
else:
    print("Распаковываем архив...")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(CITY_DIR)
    print("Готово.")

print("\nФайлы данных:")
for path in sorted(CITY_DIR.glob("*")):
    print(f"  {path.name:45s} {path.stat().st_size / 1e6:8.1f} МБ")

## 2. Граница Фрунзенского района

Берём административную границу района из OpenStreetMap и сохраняем её в файл,
чтобы не запрашивать заново при повторном запуске.

In [ ]:
# === 2. Граница Фрунзенского района ===
import osmnx as ox

BOUNDARY_FILE = DATA_DIR / "frunzensky_boundary.geojson"

if BOUNDARY_FILE.exists():
    district = gpd.read_file(BOUNDARY_FILE)
    print("Граница загружена из файла:", BOUNDARY_FILE.name)
else:
    # OSM relation 369514 — административная граница Фрунзенского района Санкт-Петербурга
    try:
        district = ox.geocode_to_gdf("R369514", by_osmid=True)
    except Exception as error:
        print("Не удалось получить границу по ID, ищем по названию:", error)
        district = ox.geocode_to_gdf("Фрунзенский район, Санкт-Петербург, Россия")
    district = district[["geometry"]].to_crs(district.estimate_utm_crs())
    district.to_file(BOUNDARY_FILE, driver="GeoJSON")
    print("Граница сохранена:", BOUNDARY_FILE.name)

print("Система координат границы:", district.crs)
print("Площадь района, км²:", round(district.geometry.union_all().area / 1e6, 1))

district.plot(color="none", edgecolor="crimson", figsize=(8, 8)).set_axis_off()
plt.title("Фрунзенский район, Санкт-Петербург")
plt.show()

## 3. Кварталы района

Загружаем готовые кварталы города и оставляем те, что попадают в границу района.
Дополнительно берём **буферную зону** вокруг района — она нужна, чтобы правильно
учитывать объекты на границе района при расчёте доступности и обеспеченности.

In [ ]:
# === 3.1. Загрузка кварталов города и отбор района ===
blocks = gpd.read_file(BLOCKS_FILE)
print("Кварталов в городе:", len(blocks))
print("Система координат кварталов:", blocks.crs)
print("Сколько столбцов (показателей) у кварталов:", blocks.shape[1])

# Приводим границу к системе координат кварталов
district = district.to_crs(blocks.crs)
district_geom = district.geometry.union_all()

# Квартал относим к району, если его центр лежит внутри границы района
district_blocks = blocks[blocks.geometry.representative_point().within(district_geom)].copy()
print("Кварталов в районе:", len(district_blocks))

In [ ]:
# === 3.2. Буферная зона вокруг района ===
BUFFER_M = 1000  # ширина буфера в метрах; можно уменьшить (например, 500) для ускорения расчётов

context_geom = district_geom.buffer(BUFFER_M)
context_blocks = blocks[blocks.geometry.representative_point().within(context_geom)].copy()
print(f"Кварталов в районе + буфер {BUFFER_M} м:", len(context_blocks))

# Сохраняем подготовленные слои, чтобы использовать их повторно
district_blocks.to_file(OUT_DIR / "frunzensky_blocks.geojson", driver="GeoJSON")
context_blocks.to_file(OUT_DIR / "frunzensky_context_blocks.geojson", driver="GeoJSON")
print("Сохранено:", (OUT_DIR / "frunzensky_blocks.geojson").name)

# Картинка: синие — кварталы района, серые — буферная зона
fig, ax = plt.subplots(figsize=(9, 9))
context_blocks.plot(ax=ax, color="#eeeeee", edgecolor="#bbbbbb", linewidth=0.2)
district_blocks.plot(ax=ax, color="#9ecae1", edgecolor="#3182bd", linewidth=0.3)
district.plot(ax=ax, color="none", edgecolor="crimson", linewidth=1.5)
ax.set_axis_off()
ax.set_title("Фрунзенский район: кварталы (синие) и буферная зона (серая)")
plt.show()

---
## ⏸ ОСТАНОВКА ПОСЛЕ ПОДГОТОВКИ КВАРТАЛОВ

На этом шаге уже готово:

- скачаны городские данные в `data/saint_petersburg/`;
- получена граница Фрунзенского района;
- определён список кварталов района и буферной зоны;
- сохранены файлы `outputs/frunzensky_blocks.geojson` и `outputs/frunzensky_context_blocks.geojson`.

**Дальше начинаются расчёты показателей.** Перед продолжением убедитесь, что карта выше
выглядит правильно и число кварталов в районе разумное.

👉 *Учебная остановка.* Здесь предполагается короткий вопрос: **какие показатели доступны
в BlocksNet и какие из них включать в исследование** — время в пути, плотность застройки,
морфотипы, обеспеченность услугами, интегральная доступность. После ответа продолжайте со
следующей ячейки.
---

## 4. Граф дорог и время в пути между кварталами

Берём **готовый** граф автомобильных дорог из архива (не строим его заново) и считаем
кратчайшее время поездки (в минутах) между центрами всех кварталов района и буферной зоны.

In [ ]:
# === 4.1. Загрузка готового графа дорог ===
with open(GRAPH_FILE, "rb") as file:
    graph = pickle.load(file)

print("Тип объекта:  ", type(graph).__name__)
print("Узлов графа:  ", f"{graph.number_of_nodes():,}")
print("Рёбер графа:  ", f"{graph.number_of_edges():,}")
print("EPSG графа:   ", graph.graph.get("crs"))
print("Вес ребра — время в пути в минутах (атрибут 'time_min')")

sample_u, sample_v, sample_data = next(iter(graph.edges(data=True)))
print(f"Пример ребра: {sample_u} -> {sample_v},  время {sample_data.get('time_min'):.2f} мин")

In [ ]:
# === 4.2. Матрица времени в пути между кварталами (минуты) ===
from blocksnet.relations import calculate_accessibility_matrix

# Расчёт может занять несколько минут: для каждого квартала ищется кратчайший путь
# по дорогам до всех остальных кварталов. Результат — время в минутах.
acc_mx = (pd.read_parquet(OUT_DIR / "accessibility_matrix_drive.parquet") if (OUT_DIR / "accessibility_matrix_drive.parquet").exists() else calculate_accessibility_matrix(context_blocks, graph)).astype("float32")  # если файл матрицы уже есть — берём его; при смене буфера удалите outputs/accessibility_matrix_drive.parquet
print("Размер матрицы (кварталов × кварталов):", acc_mx.shape)

finite = acc_mx.values[np.isfinite(acc_mx.values)]
print(
    f"Время в пути, мин:  минимум {finite.min():.1f}, "
    f"медиана {np.median(finite):.1f}, максимум {finite.max():.1f}"
)
print("Пар без маршрута (недостижимы):", int(np.isinf(acc_mx.values).sum()))

# Сохраняем результат: CSV — для чтения человеком, parquet — для быстрой повторной загрузки
acc_mx.to_csv(OUT_DIR / "accessibility_matrix_drive.csv")
acc_mx.to_parquet(OUT_DIR / "accessibility_matrix_drive.parquet")
print("Сохранено:", (OUT_DIR / "accessibility_matrix_drive.csv").name)

## 5. Плотность застройки и морфотипы

- **Плотность** — набор числовых показателей застройки квартала (FSI, GSI, MXI, этажность и др.).
- **Морфотипы** — типы застройки, полученные кластеризацией (Spacematrix) и по правилам (Strelka).

In [ ]:
# === 5.1. Показатели плотности застройки ===
from blocksnet.analysis.indicators import calculate_density_indicators

density_df = calculate_density_indicators(district_blocks)

DENSITY_COLUMNS = ["fsi", "gsi", "mxi", "l", "osr", "share_living", "share_non_living"]
density_df[DENSITY_COLUMNS].describe().round(2)

Значение показателей (кратко):

- `fsi` — отношение площади всех этажей к площади участка (плотность);
- `gsi` — доля застроенной площади;
- `mxi` — доля жилых площадей;
- `l` — средняя этажность;
- `osr` — обеспеченность открытыми пространствами;
- `share_living` / `share_non_living` — доли жилых и нежилых площадей.

In [ ]:
# === 5.2. Морфотипы застройки ===
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes

spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
strelka_df = get_strelka_morphotypes(density_df)

spacematrix_morph = spacematrix_df[["morphotype"]].rename(columns={"morphotype": "morphotype_spacematrix"})
strelka_morph = strelka_df[["morphotype"]].rename(columns={"morphotype": "morphotype_strelka"})

print("Spacematrix — типы застройки:")
print(spacematrix_morph["morphotype_spacematrix"].value_counts())
print("\nStrelka — типы застройки:")
print(strelka_morph["morphotype_strelka"].value_counts())

In [ ]:
# === 5.3. Карта морфотипов ===
fig, axes = plt.subplots(1, 2, figsize=(17, 8))
district_blocks.join(spacematrix_morph).plot(
    "morphotype_spacematrix", ax=axes[0], legend=True, cmap="tab20",
    missing_kwds={"color": "lightgrey"},
)
district_blocks.join(strelka_morph).plot(
    "morphotype_strelka", ax=axes[1], legend=True, cmap="tab20",
    missing_kwds={"color": "lightgrey"},
)
axes[0].set_title("Морфотипы Spacematrix")
axes[1].set_title("Морфотипы Strelka")
for ax in axes:
    ax.set_axis_off()
plt.show()

## 6. Обеспеченность услугами

Для каждой услуги решается задача распределения: сколько жителей каждого квартала может
доехать до услуги за нормативное время. Результат — доля спроса, закрытая в пределах
норматива доступности.

- `accessibility` — нормативное время доступности услуги (минуты);
- `demand` — норматив потребности (на 1000 жителей).

In [ ]:
# === 6.1. Базовый набор услуг ===
from blocksnet.config import service_types_config

# Услуги, важные для качества городской среды
CANDIDATE_SERVICES = [
    "school", "kindergarten", "polyclinic", "hospital", "pharmacy",
    "supermarket", "convenience", "cafe", "restaurant", "bakery",
    "park", "playground", "pitch", "swimming_pool", "theatre",
    "museum", "cinema", "university", "library", "bank", "post",
    "subway_entrance", "bus_stop", "hairdresser", "market",
    "dog_park", "mall", "multifunctional_center",
]

# Оставляем только те услуги, по которым есть данные в наборе города
SERVICES = [s for s in CANDIDATE_SERVICES if f"capacity_{s}" in district_blocks.columns]

print(f"Услуг в наборе: {len(SERVICES)}\n")
for service in SERVICES:
    info = service_types_config[service]
    print(
        f"  {service:22s} {str(info.get('name_ru')):38s} "
        f"норматив {info['demand']:>4d} на 1000 чел., доступность {info['accessibility']:>3d} мин"
    )

In [ ]:
# === 6.2. Расчёт обеспеченности (competitive provision) ===
# ВНИМАНИЕ: для большого числа услуг расчёт может занять несколько минут.
from blocksnet.analysis.provision import competitive_provision

provision_values, provision_weak_values = {}, {}
summary_rows = []

for service in SERVICES:
    info = service_types_config[service]
    accessibility = int(info["accessibility"])
    demand = int(info["demand"])

    # Для каждой услуги нужны две колонки: население (спрос) и мощности (предложение)
    data = district_blocks[["population", f"capacity_{service}"]].copy()
    data = data.rename(columns={f"capacity_{service}": "capacity"})
    data["population"] = data["population"].fillna(0)
    data["capacity"] = data["capacity"].fillna(0)

    prov_df, links_df = competitive_provision(data, acc_mx, accessibility=accessibility, demand=demand)

    provision_values[service] = prov_df["provision_strong"]
    provision_weak_values[service] = prov_df["provision_weak"]
    summary_rows.append(
        {
            "service": service,
            "name_ru": info.get("name_ru"),
            "accessibility_min": accessibility,
            "demand_per_1000": demand,
            "provision_mean": round(float(prov_df["provision_strong"].mean()), 3), "provision_weak_mean": round(float(prov_df["provision_weak"].mean()), 3),
            "share_blocks_ok": round(float((prov_df["provision_strong"] >= 0.8).mean()), 3),
        }
    )

provision_summary = pd.DataFrame(summary_rows).sort_values("provision_mean")
provision_summary

In [ ]:
# === 6.3. Карта обеспеченности по услугам ===
provision_blocks = pd.DataFrame(index=district_blocks.index)
for service, values in provision_values.items():
    provision_blocks[f"provision_{service}"] = values
    provision_blocks[f"provision_weak_{service}"] = provision_weak_values[service]
provision_blocks = provision_blocks.reindex(district_blocks.index)

# Столбчатая диаграмма: средняя обеспеченность по услугам
ax = provision_summary.set_index("service")["provision_mean"].plot.barh(
    figsize=(9, 0.35 * len(provision_summary) + 2), color="#3182bd"
)
ax.set_xlabel("Средняя обеспеченность (доля спроса, закрытого рядом)")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## 7. Интегральные показатели доступности

- `area_accessibility` — средневзвешенное по площади время до всех кварталов (насколько
  «централен» квартал);
- `mean_accessibility` / `median_accessibility` — среднее и медианное время до остальных
  кварталов (общая транспортная доступность).

In [ ]:
# === 7. Интегральные показатели доступности ===
from blocksnet.analysis.network.accessibility import (
    area_accessibility,
    mean_accessibility,
    median_accessibility,
)

area_acc = area_accessibility(acc_mx, context_blocks)
mean_acc = mean_accessibility(acc_mx)
median_acc = median_accessibility(acc_mx)

accessibility_df = district_blocks[["site_area"]].join(area_acc).join(mean_acc).join(median_acc)
accessibility_df.head()

## 8. Итоговая таблица и проверка

Собираем все показатели в один слой по кварталам района, сохраняем в `outputs/`
(GeoJSON — геометрия + атрибуты, CSV — для табличного анализа и дашборда).

In [ ]:
# === 8. Итоговая таблица показателей ===
final = district_blocks.copy()
final = final.join(density_df[DENSITY_COLUMNS])
final = final.join(spacematrix_morph)
final = final.join(strelka_morph)
final = final.join(provision_blocks)
final = final.join(accessibility_df[["area_accessibility", "mean_accessibility", "median_accessibility"]])

final["block_id"] = final.index
final.to_file(OUT_DIR / "frunzensky_indicators.geojson", driver="GeoJSON")
final.drop(columns=["geometry", "block_id"]).to_csv(OUT_DIR / "frunzensky_indicators.csv", index_label="block_id")

print("Кварталов:", len(final), "  Столбцов (показателей):", final.shape[1])
final.drop(columns="geometry").head()

In [ ]:
# === 9. Проверка результатов ===
assert len(final) == len(district_blocks), "число кварталов разошлось"
assert final.index.is_unique, "дубли индексов кварталов"

print("Все проверки пройдены.")
print("\nФайлы в папке результатов:")
for path in sorted(OUT_DIR.glob("*")):
    print(f"  {path.name:42s} {path.stat().st_size / 1e6:8.2f} МБ")

# Наглядный график: плотность застройки по кварталам
final.plot(
    "fsi", legend=True, figsize=(9, 9), cmap="viridis", missing_kwds={"color": "lightgrey"}
).set_axis_off()
plt.title("Плотность застройки (FSI) по кварталам района")
plt.show()

## 9. Краткие выводы и повторный запуск

### Что получилось
В папке `outputs/` лежат:

- `frunzensky_blocks.geojson` — кварталы района;
- `frunzensky_context_blocks.geojson` — кварталы района с буферной зоной;
- `accessibility_matrix_drive.csv` (и `.parquet`) — матрица времени в пути между кварталами, минуты;
- `frunzensky_indicators.geojson` — итоговый слой со всеми показателями;
- `frunzensky_indicators.csv` — то же в виде таблицы.

### Как перезапустить
- Повторный запуск **не требует** повторного скачивания: архив и граница уже сохранены в `data/`.
- Если нужно изменить территорию моделирования — поменяйте `BUFFER_M` (буферную зону) в разделе 3.
- Если нужно изменить набор услуг — отредактируйте список `CANDIDATE_SERVICES` в разделе 6.
- Пересчитанные результаты каждый раз перезаписывают файлы в `outputs/`.
- Матрица времени в пути переиспользуется: если файл `outputs/accessibility_matrix_drive.parquet` уже есть, она загружается, а не считается заново (при изменении `BUFFER_M` удалите этот файл).

### Следующий шаг
На основе сохранённых файлов (`outputs/*.geojson` и `outputs/frunzensky_indicators.csv`)
отдельный скрипт соберёт **одну HTML-страницу** с интерактивной картой и дашбордом
(те же показатели: графики, таблицы, выводы).